# EDA — NCBI Disease Corpus
Format: PubTator  
KB: MeSH  
Entity types: SpecificDisease, DiseaseClass, Modifier, CompositeMention

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from pathlib import Path

sns.set_theme(style='whitegrid', palette='muted')

DATA_DIR = Path('../data/raw/ncbi_disease')
TRAIN_FILE = DATA_DIR / 'NCBItrainset_corpus.txt'
DEV_FILE   = DATA_DIR / 'NCBIdevelopset_corpus.txt'
TEST_FILE  = DATA_DIR / 'NCBItestset_corpus.txt'

## 1. Parser

In [ ]:
def parse_pubtator(filepath):
    """
    Parse a PubTator-format file.
    Returns:
        docs   : list of dicts  {pmid, title, abstract}
        mentions: list of dicts {pmid, start, end, text, entity_type, mesh_ids}
    """
    docs, mentions = [], []
    current = {}

    with open(filepath, 'r') as f:
        for line in f:
            line = line.rstrip('\n')

            if '|t|' in line:
                pmid, _, title = line.split('|', 2)
                current = {'pmid': pmid, 'title': title, 'abstract': ''}

            elif '|a|' in line:
                _, _, abstract = line.split('|', 2)
                current['abstract'] = abstract
                docs.append(current)

            elif line.strip() == '':
                current = {}

            else:
                parts = line.split('\t')
                if len(parts) >= 6:
                    pmid, start, end, text, etype, mesh_raw = parts[:6]
                    mesh_ids = mesh_raw.strip().split('|')  # handles CompositeMention
                    mentions.append({
                        'pmid'       : pmid,
                        'start'      : int(start),
                        'end'        : int(end),
                        'text'       : text,
                        'entity_type': etype,
                        'mesh_ids'   : mesh_ids,
                        'is_composite': len(mesh_ids) > 1
                    })

    return docs, mentions


train_docs, train_mentions = parse_pubtator(TRAIN_FILE)
dev_docs,   dev_mentions   = parse_pubtator(DEV_FILE)
test_docs,  test_mentions  = parse_pubtator(TEST_FILE)

train_df = pd.DataFrame(train_mentions)
dev_df   = pd.DataFrame(dev_mentions)
test_df  = pd.DataFrame(test_mentions)

print(f'Train  — docs: {len(train_docs):>4}  mentions: {len(train_df):>5}')
print(f'Dev    — docs: {len(dev_docs):>4}  mentions: {len(dev_df):>5}')
print(f'Test   — docs: {len(test_docs):>4}  mentions: {len(test_df):>5}')

## 2. Quick look at the data

In [ ]:
train_df.head(10)

In [ ]:
train_df.info()
train_df['entity_type'].value_counts()

## 3. Entity type distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=False)

for ax, (df, split) in zip(axes, [(train_df, 'Train'), (dev_df, 'Dev'), (test_df, 'Test')]):
    counts = df['entity_type'].value_counts()
    sns.barplot(x=counts.values, y=counts.index, ax=ax, orient='h')
    ax.set_title(split)
    ax.set_xlabel('count')
    ax.set_ylabel('')

plt.suptitle('Entity type distribution', y=1.02, fontsize=13)
plt.tight_layout()
plt.show()

## 4. Mention length distribution (in characters)

In [ ]:
train_df['mention_len'] = train_df['text'].str.len()
train_df['mention_words'] = train_df['text'].str.split().str.len()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.histplot(train_df['mention_len'], bins=40, ax=axes[0], kde=True)
axes[0].set_title('Mention length (chars)')
axes[0].set_xlabel('characters')

sns.histplot(train_df['mention_words'], bins=20, ax=axes[1], kde=True)
axes[1].set_title('Mention length (words)')
axes[1].set_xlabel('words')

plt.tight_layout()
plt.show()

print(train_df[['mention_len', 'mention_words']].describe().round(2))

## 5. Mentions per document

In [ ]:
mentions_per_doc = train_df.groupby('pmid').size()

fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(mentions_per_doc, bins=30, ax=ax, kde=True)
ax.set_title('Mentions per document (train)')
ax.set_xlabel('number of mentions')
plt.tight_layout()
plt.show()

print(mentions_per_doc.describe().round(2))

## 6. Composite mentions

In [ ]:
n_composite = train_df['is_composite'].sum()
print(f'Composite mentions: {n_composite} ({100*n_composite/len(train_df):.1f}%)')
print()
print('Examples:')
train_df[train_df['is_composite']][['text', 'mesh_ids']].head(10)

## 7. KB coverage — how many mentions map to a valid MeSH ID?

In [ ]:
# NIL mentions have no valid MeSH ID (sometimes marked as -1 or empty)
def is_nil(mesh_ids):
    return any(m.strip() in ['-1', '', 'None', 'OMIM'] for m in mesh_ids)

train_df['is_nil'] = train_df['mesh_ids'].apply(is_nil)

nil_count = train_df['is_nil'].sum()
print(f'NIL mentions : {nil_count} ({100*nil_count/len(train_df):.1f}%)')
print(f'Mapped       : {len(train_df)-nil_count} ({100*(1-nil_count/len(train_df)):.1f}%)')

# NIL breakdown by entity type
train_df.groupby('entity_type')['is_nil'].mean().mul(100).round(1).rename('nil_%')

## 8. Most frequent mentions and MeSH IDs

In [ ]:
print('=== Top 15 most frequent mention texts ===')
print(train_df['text'].str.lower().value_counts().head(15).to_string())
print()

all_mesh = [m for ids in train_df['mesh_ids'] for m in ids if m not in ['-1','','None']]
print('=== Top 15 most frequent MeSH IDs ===')
print(pd.Series(Counter(all_mesh)).sort_values(ascending=False).head(15).to_string())

## 9. Unique MeSH IDs (KB size)

In [ ]:
unique_mesh_train = set(m for ids in train_df['mesh_ids'] for m in ids if m not in ['-1','','None'])
unique_mesh_test  = set(m for ids in test_df['mesh_ids']  for m in ids if m not in ['-1','','None'])

print(f'Unique MeSH IDs in train : {len(unique_mesh_train)}')
print(f'Unique MeSH IDs in test  : {len(unique_mesh_test)}')
print(f'Test IDs not seen in train (zero-shot): {len(unique_mesh_test - unique_mesh_train)}')

## 10. Surface form variation — how many texts map to the same MeSH ID?

In [ ]:
# For each MeSH ID, how many distinct surface forms exist?
mesh_to_surfaces = {}
for _, row in train_df[~train_df['is_nil']].iterrows():
    for mid in row['mesh_ids']:
        mesh_to_surfaces.setdefault(mid, set()).add(row['text'].lower())

surface_counts = pd.Series({k: len(v) for k, v in mesh_to_surfaces.items()})

fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(surface_counts, bins=30, ax=ax)
ax.set_title('Surface form variety per MeSH ID (train)')
ax.set_xlabel('distinct mention texts per MeSH ID')
plt.tight_layout()
plt.show()

print(f'Mean surface forms per MeSH ID: {surface_counts.mean():.2f}')
print(f'Max surface forms for one ID  : {surface_counts.max()}')
print()
print('Most ambiguous MeSH IDs:')
top_ids = surface_counts.sort_values(ascending=False).head(5)
for mid, cnt in top_ids.items():
    print(f'  {mid}: {cnt} forms → {list(mesh_to_surfaces[mid])[:5]}')

## 11. Summary stats

In [ ]:
summary = {
    'split'            : ['train', 'dev', 'test'],
    'documents'        : [len(train_docs), len(dev_docs), len(test_docs)],
    'mentions'         : [len(train_df), len(dev_df), len(test_df)],
    'unique_mesh_ids'  : [
        len(set(m for ids in train_df['mesh_ids'] for m in ids if m not in ['-1','','None'])),
        len(set(m for ids in dev_df['mesh_ids']   for m in ids if m not in ['-1','','None'])),
        len(set(m for ids in test_df['mesh_ids']  for m in ids if m not in ['-1','','None'])),
    ],
    'composite_%'      : [
        round(100*train_df['is_composite'].mean(), 1),
        round(100*dev_df['is_composite'].mean(), 1),
        round(100*test_df['is_composite'].mean(), 1),
    ],
}

pd.DataFrame(summary).set_index('split')